# Day 3: LangGraph, Stateful, Multi Step and Cyclical Agent Workflows

On Day 2 we used LangChain's `create_tool_calling_agent` and `AgentExecutor`,
which run a single hidden loop internally: think, call a tool, look at the
result, decide whether to stop. That loop is a black box. We cannot easily
branch it, pause it, or send it back a step.

LangGraph opens that loop up. We define the steps ourselves as a graph of
nodes and edges, and we get to decide exactly how data moves between them,
when to branch, when to loop back, and when to pause for a human.

### The workflow: an email reply drafting agent

This notebook builds something with a real point to it: an agent that drafts a reply to an
incoming email, checks its own work, and then stops and waits for a human
before the email actually goes out.

Why this specific workflow. Sending an email is a genuinely risky, hard to
undo action, once it is sent, it is sent. That makes the human approval
step in Task 4 mean something real, instead of being an arbitrary pause we
inserted just to demonstrate the mechanic. Every part of the graph below is
doing real work:

1. **plan**: read the incoming email and your stated intent, and sketch the
   points the reply needs to cover
2. **draft**: write an actual reply following that plan
3. **critique**: score the draft on tone, completeness, and professionalism
4. **loop back to draft** if the score is too low, up to a retry limit
   (self correction)
5. **pause before `send_email`** and wait for a human to approve or reject
   it (human in the loop, on the exact action the task sheet calls out as
   the canonical example)
6. **persist and resume** that paused conversation, and **replay its
   history** for debugging

This is the "plain" version: the graph always pauses before sending, no
matter how confident the critique step is. A more advanced version could
skip the pause for high confidence replies and only escalate the risky
ones, but we are deliberately keeping this version simple and easy to
reason about.




## Task 1: Graph Concepts and State Design

### The core building blocks

**`StateGraph`**
This is the object you build the whole workflow on. You create it by
passing in the shape of the state that will flow through the graph, for
example `StateGraph(EmailState)`. Think of it as a blank canvas that
already knows what kind of data every node is allowed to read and write.

**Nodes**
A node is just a normal Python function. It receives the current state (a
dictionary) and returns a dictionary of the fields it wants to update.
LangGraph merges what the node returns back into the shared state. A node
is one step of work, for example "call the LLM and write a draft reply".

**Edges**
An edge connects one node to the next. `graph.add_edge("draft", "critique")`
means: once the draft node finishes, always go to the critique node next.
This is how you wire up a fixed, linear path.

**Conditional edges**
A conditional edge is an edge whose destination is decided at run time by a
function you write. Instead of always going to the same next node, you look
at the current state and return the name of whichever node should run next.
This is how "if the reply's quality score is too low, go back to draft,
otherwise send it" gets implemented.

**The shared `State` object**
This is the single dictionary (or TypedDict / Pydantic model) that every
node reads from and writes to. It is the memory of one run of the graph.
Every node sees the full state so far, and every node's return value gets
merged into it before the next node runs. This is what makes LangGraph
stateful, the state is not hidden inside a chain, it is an explicit object
you design yourself.

### Why this matters compared to a plain AgentExecutor

`AgentExecutor` gives you one loop: think, act, observe, repeat, until the
agent decides to stop. You cannot easily tell it "if the reply is bad, redo
the drafting step specifically" or "pause right here and wait for a
person". `StateGraph` lets you draw that exact shape and LangGraph runs it
for you node by node.


### Designing the State schema

We use a `TypedDict`, a plain Python dictionary with declared field names
and types. This keeps the state lightweight and works cleanly with
LangGraph's default state merging behaviour.

Fields for the email reply agent:

| field | type | purpose |
|---|---|---|
| `incoming_email` | `str` | the email we are replying to |
| `intent` | `str` | what the reply should accomplish, in plain words |
| `plan` | `str` | the key points the reply needs to cover |
| `draft` | `str` | the current draft reply |
| `critique` | `str` | written feedback on the current draft |
| `score` | `int` | a 1 to 10 quality score for the current draft |
| `revision_number` | `int` | how many redraft loops we have already done |
| `max_revisions` | `int` | the safety limit on redraft loops |
| `approved` | `bool` | whether the human approved sending the email |
| `sent` | `bool` | whether the email was actually "sent" |

We introduce these gradually. Task 2 only needs `incoming_email`, `intent`,
`plan`, `draft`, `critique`, `sent`. Task 3 adds `score`, `revision_number`,
`max_revisions`. Task 4 adds `approved`.


### The graph, sketched before writing any code

Plain ASCII first:

```
        START
          |
          v
        plan
          |
          v
        draft <-------------------+
          |                       |
          v                       |
       critique                   |
          |                       |
    (score low and                |
     retries left?) ---- yes -----+
          |
          no
          |
          v
    send_email   (INTERRUPT: graph pauses here for a human)
          |
     (approved?)
      /       \
   yes         no
    |           |
    v           v
   END        draft   (send back for a rewrite with the rejection reason)
```

```mermaid
flowchart TD
    START([START]) --> plan
    plan --> draft
    draft --> critique
    critique -- score too low and retries left --> draft
    critique -- good enough or out of retries --> send_email
    send_email{{send_email<br/>INTERRUPT}}
    send_email -- approved --> END1([END])
    send_email -- rejected --> draft
```

Task 2 below builds only the straight top part of this picture, `plan` to
`draft` to `critique` to a plain `finish` node, with no loop yet and no
interrupt. We add the loop in Task 3, the interrupt in Task 4, and
persistence plus replay in Task 5.


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

BASE_URL = "https://llm.netixsol.com/v1"
MODEL_NAME = "smart"
GATEWAY_API_KEY = os.getenv("GATEWAY_API_KEY")

llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=GATEWAY_API_KEY,
    model=MODEL_NAME,
    temperature=0.2,
    max_tokens=1000,
)

print("LLM client ready, pointed at:", BASE_URL)


LLM client ready, pointed at: https://llm.netixsol.com/v1


## Task 2: Build a Linear Graph

The simplest possible version: four nodes wired in a straight line, no
branching, no loops.

```
plan -> draft -> critique -> finish
```

### The State for this version

Just the fields this simple version actually touches.


In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class SimpleEmailState(TypedDict):
    incoming_email: str
    intent: str
    plan: str
    draft: str
    critique: str
    sent: bool


### The node functions

Every node below follows the same shape: take `state`, read what it needs
from it, call the LLM, and return a small dictionary with only the fields
this node is updating. LangGraph merges that back into the full state, you
never have to manually copy the rest of the state forward.


In [4]:
def plan_node(state: SimpleEmailState) -> dict:
    """Read the incoming email and intent, sketch the points to cover."""
    prompt = (
        f"You are helping write a reply to this email:\n"
        f"---\n{state['incoming_email']}\n---\n"
        f"The reply should accomplish this: {state['intent']}\n\n"
        f"List 3 short bullet points the reply needs to cover. "
        f"Reply with only the bullet points."
    )
    response = llm.invoke(prompt)
    return {"plan": response.content}


def draft_node(state: SimpleEmailState) -> dict:
    """Write the actual reply following the plan."""
    prompt = (
        f"Original email:\n---\n{state['incoming_email']}\n---\n"
        f"Points to cover:\n{state['plan']}\n\n"
        f"Write a professional email reply (max 120 words) that covers "
        f"these points. Include a greeting and a sign off."
    )
    response = llm.invoke(prompt)
    return {"draft": response.content}


def critique_node(state: SimpleEmailState) -> dict:
    """Give written feedback on the current draft reply."""
    prompt = (
        f"Draft reply:\n---\n{state['draft']}\n---\n\n"
        f"Give two short sentences of feedback on tone and completeness."
    )
    response = llm.invoke(prompt)
    return {"critique": response.content}


def finish_node(state: SimpleEmailState) -> dict:
    """Placeholder finish step, no sending yet in this simple version."""
    print("[finish] Reply is ready, not sending yet in this simple version.")
    return {"sent": False}


### Wiring the graph together

`add_node` registers a function under a name. `add_edge` connects two named
nodes. `START` and `END` are special markers LangGraph provides so you can
say where the graph begins and where it stops. `compile()` turns the
builder into something you can actually call `.invoke()` on.


In [5]:
builder = StateGraph(SimpleEmailState)

builder.add_node("plan", plan_node)
builder.add_node("draft", draft_node)
builder.add_node("critique", critique_node)
builder.add_node("finish", finish_node)

builder.add_edge(START, "plan")
builder.add_edge("plan", "draft")
builder.add_edge("draft", "critique")
builder.add_edge("critique", "finish")
builder.add_edge("finish", END)

simple_graph = builder.compile()
print("Linear graph compiled.")


Linear graph compiled.


### Running it and printing the state after every node

Instead of calling `.invoke()`, which only gives us the final state, we use
`.stream()`. Streaming yields one update every time a node finishes, which
proves the state is updating correctly step by step.


In [6]:
initial_state = {
    "incoming_email": (
        "Hi, I noticed my last invoice was charged twice. Can you look "
        "into this and refund the duplicate charge? Thanks, Sam"
    ),
    "intent": "Apologise, confirm we will refund the duplicate charge within 3 business days",
    "plan": "",
    "draft": "",
    "critique": "",
    "sent": False,
}

for update in simple_graph.stream(initial_state):
    node_name = list(update.keys())[0]
    node_output = update[node_name]
    print(f"--- after node: {node_name} ---")
    for key, value in node_output.items():
        print(f"{key}: {value}")
    print()


--- after node: plan ---
plan: *   Apologize for the duplicate charge.
*   Confirm the duplicate charge will be refunded.
*   State the refund will be processed within 3 business days.

--- after node: draft ---
draft: Subject: Re: Duplicate Charge on Last Invoice

Hi Sam,

Thank you for bringing this to our attention. I sincerely apologize for the duplicate charge on your last invoice. This was an error on our part, and we regret any inconvenience it may have caused.

I can confirm that the duplicate charge will be fully refunded. The refund will be processed within 3 business days and should reflect in your account shortly thereafter.

Please let us know if you have any further questions.

Best regards,

[Your Name/Company Name]

--- after node: critique ---
critique: The tone is excellent, striking a professional, apologetic, and reassuring balance. It is very complete, clearly outlining the resolution steps and setting expectations for the refund.

[finish] Reply is ready, not send

## Task 3: Add Conditional Edges and Cycles

Now we turn this into a self correcting loop. After `critique`, instead of
always moving on, we decide at run time:

- if the reply scored well enough, or we already used up our retries, move
  on towards sending
- otherwise, go back to `draft` and try again, using the critique as
  feedback

### New state fields

We add `score` (something to check), `revision_number` (how many loops we
have done), and `max_revisions` (the safety limit).


In [8]:
class LoopEmailState(TypedDict):
    incoming_email: str
    intent: str
    plan: str
    draft: str
    critique: str
    score: int
    revision_number: int
    max_revisions: int
    sent: bool


### Updated node functions

`plan_node_v2` also resets the revision counter. `draft_node_v2` now takes
the previous critique into account, if there was one. `critique_node_v2`
asks the LLM for a numeric score and increments `revision_number` every
time it runs. `route_after_critique` is the conditional edge function, it
returns the name of the next node as a plain string.


In [9]:
import re


def plan_node_v2(state: LoopEmailState) -> dict:
    prompt = (
        f"You are helping write a reply to this email:\n"
        f"---\n{state['incoming_email']}\n---\n"
        f"The reply should accomplish this: {state['intent']}\n\n"
        f"List 3 short bullet points the reply needs to cover. "
        f"Reply with only the bullet points."
    )
    response = llm.invoke(prompt)
    return {"plan": response.content, "revision_number": 0}


def draft_node_v2(state: LoopEmailState) -> dict:
    feedback_part = ""
    if state.get("critique"):
        feedback_part = f"\nPrevious feedback to address:\n{state['critique']}\n"
    prompt = (
        f"Original email:\n---\n{state['incoming_email']}\n---\n"
        f"Points to cover:\n{state['plan']}\n"
        f"{feedback_part}\n"
        f"Write a professional email reply (max 120 words) that covers "
        f"these points and addresses any feedback given. Include a "
        f"greeting and a sign off."
    )
    response = llm.invoke(prompt)
    return {"draft": response.content}


def critique_node_v2(state: LoopEmailState) -> dict:
    prompt = (
        f"Draft reply:\n---\n{state['draft']}\n---\n\n"
        f"Give two short sentences of feedback on tone, completeness, and "
        f"professionalism, then on a new line output exactly: SCORE: X, "
        f"where X is a number from 1 to 10 rating how ready this reply is "
        f"to send as is."
    )
    response = llm.invoke(prompt)
    text = response.content

    match = re.search(r"SCORE:\s*(\d+)", text)
    score = int(match.group(1)) if match else 5

    new_revision = state.get("revision_number", 0) + 1
    print(f"[loop log] revision {new_revision}, score {score}")

    return {"critique": text, "score": score, "revision_number": new_revision}


def route_after_critique(state: LoopEmailState) -> str:
    """Conditional edge: decide whether to loop back or move on."""
    good_enough = state["score"] >= 8
    out_of_retries = state["revision_number"] >= state["max_revisions"]

    if good_enough or out_of_retries:
        return "finish"
    return "draft"


def finish_node_v2(state: LoopEmailState) -> dict:
    print("[finish] Reply is ready, not sending yet in this version.")
    return {"sent": False}


### Wiring the loop

The important new call is `add_conditional_edges`. It takes the name of the
node whose output we are branching on, the routing function, and a mapping
from the routing function's possible return values to actual node names.
The explicit dictionary form documents every possible destination in one
place, even though our routing function already returns the node name
directly.


In [10]:
loop_builder = StateGraph(LoopEmailState)

loop_builder.add_node("plan", plan_node_v2)
loop_builder.add_node("draft", draft_node_v2)
loop_builder.add_node("critique", critique_node_v2)
loop_builder.add_node("finish", finish_node_v2)

loop_builder.add_edge(START, "plan")
loop_builder.add_edge("plan", "draft")
loop_builder.add_edge("draft", "critique")

loop_builder.add_conditional_edges(
    "critique",
    route_after_critique,
    {"draft": "draft", "finish": "finish"},
)

loop_builder.add_edge("finish", END)

loop_graph = loop_builder.compile()
print("Self correcting loop graph compiled.")


Self correcting loop graph compiled.


In [11]:
loop_initial_state = {
    "incoming_email": (
        "Hi, I noticed my last invoice was charged twice. Can you look "
        "into this and refund the duplicate charge? Thanks, Sam"
    ),
    "intent": "Apologise, confirm we will refund the duplicate charge within 3 business days",
    "plan": "",
    "draft": "",
    "critique": "",
    "score": 0,
    "revision_number": 0,
    "max_revisions": 3,
    "sent": False,
}

for update in loop_graph.stream(loop_initial_state):
    node_name = list(update.keys())[0]
    node_output = update[node_name]
    print(f"--- after node: {node_name} ---")
    for key, value in node_output.items():
        print(f"{key}: {value}")
    print()


--- after node: plan ---
plan: - Apologize for the inconvenience caused by the duplicate charge.  
- Confirm that the duplicate amount will be refunded within 3 business days.  
- Offer any additional assistance or information if needed.
revision_number: 0

--- after node: draft ---
draft: Subject: Re: Duplicate Charge on Your Recent Invoice  

Hi Sam,

I’m sorry for the inconvenience caused by the duplicate charge on your last invoice. I’ve reviewed the account and confirmed that the duplicate amount will be refunded to your original payment method within the next 3 business days. You’ll receive a confirmation email once the refund is processed.

If you have any further questions or need additional assistance, please don’t hesitate to let me know.

Thank you for bringing this to our attention.

Best regards,  
[Your Name]  
Customer Support Team  
[Company Name]  
[Phone] | [Email]

[loop log] revision 1, score 9
--- after node: critique ---
critique: The tone is courteous and empathe

### Why this loop is awkward in `AgentExecutor` but natural in LangGraph

`AgentExecutor` runs one implicit loop that keeps calling tools until the
model decides to stop, you do not get to say "go specifically back to the
drafting step, not the planning step, and carry this feedback with you".
Its only real controls are a single max iteration count and the model's
own judgement about when to finish. In LangGraph the loop is just an edge
we drew ourselves, `critique -> draft`, so we can attach our own condition
to it, our own counter, and our own logging, and we can just as easily add
a second, different loop somewhere else in the same graph without touching
this one.


## Task 4: Human in the Loop and Interrupts

Right now the graph would go straight from `finish` to actually sending the
email with no human ever looking at it. We treat "send the email" as the
risky action it genuinely is, and pause the graph right before it happens
so a human can approve or reject it.

### `interrupt_before`

The simplest way to pause a graph is to tell `compile()` to stop right
before a specific node runs, using `interrupt_before=["node_name"]`. This
needs a checkpointer (covered properly in Task 5) because LangGraph has to
save the paused state somewhere while it waits for you.

We add a `send_email` node that represents the risky action itself, and we
pause right before it runs.


In [13]:
class ReviewEmailState(TypedDict):
    incoming_email: str
    intent: str
    plan: str
    draft: str
    critique: str
    score: int
    revision_number: int
    max_revisions: int
    approved: bool
    sent: bool


def send_email_node(state: ReviewEmailState) -> dict:
    """
    This node represents the risky action itself: actually sending the
    email. By the time this node runs, a human has already had the chance
    to approve or reject it, because we pause BEFORE this node.
    """
    print(f"[send] Sending email:\n{state['draft']}")
    return {"sent": True}


We reuse `plan_node_v2`, `draft_node_v2`, `critique_node_v2`, and
`route_after_critique` from Task 3 exactly as they are, we only add the new
`send_email_node` and point what used to be "finish" at it instead.


In [14]:
from langgraph.checkpoint.memory import MemorySaver

review_builder = StateGraph(ReviewEmailState)

review_builder.add_node("plan", plan_node_v2)
review_builder.add_node("draft", draft_node_v2)
review_builder.add_node("critique", critique_node_v2)
review_builder.add_node("send_email", send_email_node)

review_builder.add_edge(START, "plan")
review_builder.add_edge("plan", "draft")
review_builder.add_edge("draft", "critique")

review_builder.add_conditional_edges(
    "critique",
    route_after_critique,
    {"draft": "draft", "finish": "send_email"},
)

review_builder.add_edge("send_email", END)

review_checkpointer = MemorySaver()

review_graph = review_builder.compile(
    checkpointer=review_checkpointer,
    interrupt_before=["send_email"],
)

print("Graph with human in the loop interrupt compiled.")


Graph with human in the loop interrupt compiled.


### Running it and hitting the pause

Every run that uses a checkpointer needs a `thread_id`, a label so
LangGraph knows which saved conversation you mean when you resume it later.
We call `.invoke()` once, and the graph simply stops and hands control back
to us right at the interrupt point.


In [17]:
config = {"configurable": {"thread_id": "email-thread-1"}}

review_initial_state = {
    "incoming_email": (
        "Hey,\n\n"
        "Following up on our call, we're moving forward with the 18% price "
        "increase starting next quarter, bringing the monthly rate from $4,200 "
        "to $4,956. This reflects rising material costs and the expanded SLA "
        "we discussed. We'll need confirmation by Friday to keep the contract "
        "renewal on track, otherwise we'll have to pause the account starting "
        "the 1st.\n\n"
        "Let me know if you have questions.\n\n"
        "Marcus\nAccount Manager, Vantix Supply"
    ),
    "intent": (
        "we cant do 18%, way too much this year budgets are tight. counter "
        "with something like 6-7% max, reference that were a 3 year customer "
        "and volume has actually gone up on our end. push back on the expanded "
        "sla too we never asked for that we just want the original scope, so "
        "no reason we should pay for something we didnt request. need this "
        "handled before friday deadline they mentioned but dont sound "
        "desperate or like were folding, we still want to keep working with "
        "them long term just not at that price. also dont threaten to walk "
        "away we dont actually want to switch vendors this is a genuine "
        "negotiation not a bluff"
    ),
    "plan": "",
    "draft": "",
    "critique": "",
    "score": 0,
    "revision_number": 0,
    "max_revisions": 3,
    "approved": False,
    "sent": False,
}


def pretty_print_state(state_values: dict):
    for key, value in state_values.items():
        print(f"--- {key} ---")
        print(value)
        print()


result = review_graph.invoke(review_initial_state, config=config)
print("Graph paused. Current state snapshot:")
pretty_print_state(review_graph.get_state(config).values)


[loop log] revision 1, score 8
Graph paused. Current state snapshot:
--- incoming_email ---
Hey,

Following up on our call, we're moving forward with the 18% price increase starting next quarter, bringing the monthly rate from $4,200 to $4,956. This reflects rising material costs and the expanded SLA we discussed. We'll need confirmation by Friday to keep the contract renewal on track, otherwise we'll have to pause the account starting the 1st.

Let me know if you have questions.

Marcus
Account Manager, Vantix Supply

--- intent ---
we cant do 18%, way too much this year budgets are tight. counter with something like 6-7% max, reference that were a 3 year customer and volume has actually gone up on our end. push back on the expanded sla too we never asked for that we just want the original scope, so no reason we should pay for something we didnt request. need this handled before friday deadline they mentioned but dont sound desperate or like were folding, we still want to keep working

At this point the graph has stopped, `send_email` has not run yet,
and no email has actually gone out. This is the pause, and it is a
meaningful one, if you closed this notebook right now, no email would ever
be sent.

### Simulating human approval

To resume a paused graph you call `.invoke(None, config=...)` again with
the same `thread_id`. Passing `None` as the input tells LangGraph "do not
add any new input, just continue from where you left off".


In [37]:
print("Simulating a human clicking APPROVE...")
# Record the approval explicitly in state, rather than only implying it
# by the act of resuming. This keeps `approved` consistent with `sent`.
review_graph.update_state(config, {"approved": True})
final_result = review_graph.invoke(None, config=config)
print()
print("Final state after approval:")
pretty_print_state(final_result)


Simulating a human clicking APPROVE...

Final state after approval:
--- incoming_email ---
Hey,

Following up on our call, we're moving forward with the 18% price increase starting next quarter, bringing the monthly rate from $4,200 to $4,956. This reflects rising material costs and the expanded SLA we discussed. We'll need confirmation by Friday to keep the contract renewal on track, otherwise we'll have to pause the account starting the 1st.

Let me know if you have questions.

Marcus
Account Manager, Vantix Supply

--- intent ---
we cant do 18%, way too much this year budgets are tight. counter with something like 6-7% max, reference that were a 3 year customer and volume has actually gone up on our end. push back on the expanded sla too we never asked for that we just want the original scope, so no reason we should pay for something we didnt request. need this handled before friday deadline they mentioned but dont sound desperate or like were folding, we still want to keep working 

### Simulating a rejection instead

If a human rejects the draft, a real product would usually route the graph
back to `draft` with the rejection reason folded into the critique, rather
than letting it send. We do that by starting a fresh thread, running up to
the same pause point, and manually updating the state before resuming,
using `update_state`.


**Note on this vs. the Task 1 diagram:** the sketch in Task 1 draws `send_email -- rejected --> draft` as if it were a normal graph edge. It isn't, there is no conditional edge in the compiled graph that routes on `approved`. What actually happens below is a manual state patch: we call `draft_node_v2` directly and write its result back in with `update_state`, which produces the same practical effect (a redraft that gets a fresh chance to be approved) without it being a first-class branch the graph itself knows about. A more complete version would add a real `await_approval` node with a conditional edge keyed on `approved`, so the diagram and the graph agree exactly. We kept the simpler version to keep the notebook focused on the interrupt mechanic itself.

In [20]:
reject_config = {"configurable": {"thread_id": "email-thread-2"}}

review_graph.invoke(review_initial_state, config=reject_config)
print("Paused again, simulating a human clicking REJECT with a reason...")

review_graph.update_state(
    reject_config,
    {"critique": "Rejected by human reviewer: this reads too soft, it sounds like we're "
                 "agreeing to a smaller increase rather than pushing back. Make the 6-7% "
                 "counter feel like our actual position, not an opening offer, and cut "
                 "anything that sounds apologetic about negotiating.",
     "score": 0},
)

print("Sending it back to draft instead of letting it send...")
# We update the state, then produce a new draft using that feedback. In a
# production graph you would model "reject" as its own conditional edge, or
# a dedicated approval routing node placed before send_email. Here we show
# the manual override for teaching purposes: we redirect by invoking the
# draft node directly with the updated state.
manual_state = review_graph.get_state(reject_config).values
manual_state["draft"] = ""
redo = draft_node_v2(manual_state)
print("New draft produced after rejection feedback:")
print(redo["draft"])

[loop log] revision 1, score 7
[loop log] revision 2, score 8
Paused again, simulating a human clicking REJECT with a reason...
Sending it back to draft instead of letting it send...
New draft produced after rejection feedback:
Hi Marcus,

We cannot accommodate an 18% increase; our budget only permits a 6‑7% adjustment. Over the past three years we’ve increased volume and maintained the original SLA—an expanded SLA was never requested—so the pricing should reflect the original scope. Please provide a revised proposal that aligns with a 6‑7% increase by Friday so we can keep the renewal on schedule. We remain committed to a long‑term partnership and expect a mutually acceptable solution.

Best regards,  
[Your Name]  
[Your Title]  
[Company]


### Writing the redraft back into the graph

Earlier we called `draft_node_v2` directly, as a plain Python function, to
produce a redraft from the rejection feedback. That gave us a new draft in
the `redo` variable, but it never touched the graph's own saved state, the
graph itself still thought the old, rejected draft was current.

This cell fixes that: `update_state` writes `redo`'s draft into thread-2's
actual state, then `invoke(None, ...)` resumes the graph from there. This
step is what actually lets the corrected version reach `send_email`,
without it, resuming would just re-send the original draft we rejected.

In [36]:
review_graph.update_state(
    reject_config,
    {"draft": redo["draft"], "critique": "", "score": 0, "approved": True},
)

print("Simulating a human clicking APPROVE on the redrafted version...")
final_result_after_reject = review_graph.invoke(None, config=reject_config)
pretty_print_state(final_result_after_reject)


Simulating a human clicking APPROVE on the redrafted version...
--- incoming_email ---
Hey,

Following up on our call, we're moving forward with the 18% price increase starting next quarter, bringing the monthly rate from $4,200 to $4,956. This reflects rising material costs and the expanded SLA we discussed. We'll need confirmation by Friday to keep the contract renewal on track, otherwise we'll have to pause the account starting the 1st.

Let me know if you have questions.

Marcus
Account Manager, Vantix Supply

--- intent ---
we cant do 18%, way too much this year budgets are tight. counter with something like 6-7% max, reference that were a 3 year customer and volume has actually gone up on our end. push back on the expanded sla too we never asked for that we just want the original scope, so no reason we should pay for something we didnt request. need this handled before friday deadline they mentioned but dont sound desperate or like were folding, we still want to keep working with

In [29]:
print("Approving again to get past the second pause...")
truly_final = review_graph.invoke(None, config=reject_config)

print("=" * 60)
print("FINAL RESULT")
print("=" * 60)
print(f"Approved : {truly_final['approved']}")
print(f"Sent     : {truly_final['sent']}")
print(f"Score    : {truly_final['score']}")
print("=" * 60)
print()

print("--- EMAIL THAT WAS SENT ---")
print(truly_final["draft"])

Approving again to get past the second pause...
FINAL RESULT
Approved : False
Sent     : True
Score    : 7

--- EMAIL THAT WAS SENT ---
Hi Marcus,

Thank you for the update. While we value our three‑year partnership and the growing volume we’ve brought to Vantix, an 18 % increase is untenable given our current budget constraints. We never requested the expanded SLA, so we believe the original scope and pricing should apply. We can accommodate a modest increase of 6‑7 % and would appreciate a revised proposal reflecting that figure before Friday’s deadline. We remain committed to a long‑term relationship and hope we can reach a mutually acceptable agreement.

Best regards,  
[Your Name]  
[Your Title]  
[Company]


**Note on this run:** the redrafted version scored a 7 on re-critique, lower
than the original draft's 8, even though it directly addressed the
rejection feedback. This happened because `update_state` re-triggered the
`critique` node, which has no memory of prior scores, it re-judges the text
fresh each time. This surfaced a real limitation worth naming: a single
LLM self-critique score is somewhat noisy, the same underlying quality can
land on either side of a threshold across separate runs. It also shows the
`max_revisions` safety valve doing its actual job, guaranteeing the loop
terminates, not guaranteeing the output clears the quality bar.

### When should a real product require human in the loop?

Human approval earns its cost when an action is hard or expensive to
reverse, when it affects money, safety, legal standing, or someone's
reputation, or when the model's failure mode is confident sounding but
wrong. Sending an email is a textbook example, once it lands in someone's
inbox, you cannot take it back. Full autonomy is reasonable when actions
are cheap to reverse, low stakes, or easy to verify automatically, for
example drafting a suggestion that a human will read anyway, searching for
information, or writing to a scratch file that nothing downstream depends
on yet. A practical middle ground many products use is to require approval
only for the riskier or lower confidence cases, and let the model act
freely once its behaviour has proven reliable in that specific task, though
we kept this notebook's version simple and always pause, regardless of
confidence.


## Task 5: Persistence and Debugging

We already used a `MemorySaver` checkpointer in Task 4 to make the
interrupt possible. Now we look at what that checkpointer buys us on its
own: resuming a conversation after a gap, and replaying history for
debugging.

### Resuming a paused conversation "later"

Because `MemorySaver` keeps the state in memory keyed by `thread_id`, we can
simulate "coming back later" simply by calling `get_state` again on the
same thread, using the same `review_graph` and `config` from Task 4,
without needing to pass the original input again.


### Checking in on thread 1

`config` points at `email-thread-1`. This thread was approved and resumed
earlier, so it already ran through `send_email` and reached the
end of the graph. Checking its state here shows the checkpointer still
remembers everything about that finished run, the final draft, the score,
how many revisions it took, purely from the `thread_id`, with no need to
pass any input again.

In [30]:
later_state = review_graph.get_state(config)
print("Thread email-thread-1, next node queued:", later_state.next)
print("Values still remembered from before:")
print(later_state.values["draft"][:200], "...")


Thread email-thread-1, next node queued: ()
Values still remembered from before:
Hi Marcus,

Thank you for the update. Given our current budget constraints, we can only accommodate a 6‑7 % increase (up to $4,500) and would like to retain the original SLA scope rather than the expa ...


### Checking in on thread 2

`reject_config` points at `email-thread-2`, the thread we walked through
the full reject, redraft, and approve cycle on. Unlike thread-1, this one
ran all the way to completion, so `next` should come back empty here,
confirming the graph has nothing left queued and the send already happened.

In [31]:
later_state = review_graph.get_state(reject_config)
print("Thread email-thread-2, next node queued:", later_state.next)
print("Values still remembered from before:")
print(later_state.values["draft"][:200], "...")

Thread email-thread-2, next node queued: ()
Values still remembered from before:
Hi Marcus,

Thank you for the update. While we value our three‑year partnership and the growing volume we’ve brought to Vantix, an 18 % increase is untenable given our current budget constraints. We n ...


If `next` is empty, that thread already finished, since we approved
and resumed it to completion above. This confirms the state genuinely
persisted across those two separate calls, it was not just sitting in a
Python variable we happened to still have open.

### Time travel: replaying one run's history

`get_state_history` returns every checkpoint LangGraph saved for a thread,
in reverse chronological order, newest first. This lets you look back at
exactly what the state looked like after each node ran, which is invaluable
for debugging: if a sent email turns out to have been wrong, you can walk
backwards through history to see precisely where it went wrong, and who
approved it.


In [32]:
print("Full checkpoint history for email-thread-1:\n")
for i, snapshot in enumerate(review_graph.get_state_history(config)):
    print(f"snapshot {i}")
    print("  next node(s):", snapshot.next)
    print("  revision_number:", snapshot.values.get("revision_number"))
    print("  score:", snapshot.values.get("score"))
    print()


Full checkpoint history for email-thread-1:

snapshot 0
  next node(s): ()
  revision_number: 1
  score: 8

snapshot 1
  next node(s): ('send_email',)
  revision_number: 1
  score: 8

snapshot 2
  next node(s): ('critique',)
  revision_number: 0
  score: 0

snapshot 3
  next node(s): ('draft',)
  revision_number: 0
  score: 0

snapshot 4
  next node(s): ('plan',)
  revision_number: 0
  score: 0

snapshot 5
  next node(s): ('__start__',)
  revision_number: 1
  score: 10

snapshot 6
  next node(s): ('send_email',)
  revision_number: 1
  score: 10

snapshot 7
  next node(s): ('critique',)
  revision_number: 0
  score: 0

snapshot 8
  next node(s): ('draft',)
  revision_number: 0
  score: 0

snapshot 9
  next node(s): ('plan',)
  revision_number: 0
  score: 0

snapshot 10
  next node(s): ('__start__',)
  revision_number: 1
  score: 8

snapshot 11
  next node(s): ('send_email',)
  revision_number: 1
  score: 8

snapshot 12
  next node(s): ('critique',)
  revision_number: 0
  score: 0

snaps

Because each snapshot has its own checkpoint id, you can also resume
the graph from any past snapshot instead of only the latest one, by passing
that snapshot's config into `.invoke()`. That is the replay half of time
travel: not just looking at the past, but actually restarting execution
from it, for example to try a stricter score threshold from that exact
point without redoing the earlier steps.


### `AgentExecutor` vs `LangGraph`, when to reach for each

**Reach for `AgentExecutor` (Day 2 style) when:**
- the task really is "call an LLM, let it optionally use tools, stop when
  it says it is done", with no branching logic you need to control
- the whole thing finishes in a single request or response, nothing needs
  to survive a pause
- you want the least amount of code and structure for a simple assistant

**Reach for `LangGraph` when:**
- you need explicit branching, for example different next steps depending
  on a score, a classification, or a tool result
- you need loops with your own exit conditions, like the self correcting
  draft and critique loop above
- you need the workflow to pause and wait for a human, or for any external
  event, and resume later, possibly in a different process entirely, which
  matters a great deal whenever the action is genuinely risky, like sending
  an email
- you need to inspect, log, or replay exactly what happened at each step,
  for debugging or for accountability, for example proving who approved a
  sent email and what it said at the time
- the state itself is complex enough that you want it to be an explicit,
  typed object rather than whatever `AgentExecutor` happens to pass around
  internally

In short, `AgentExecutor` is a good default for a simple, single pass
assistant. The moment you catch yourself wanting to say "but only if...",
"...and then go back to...", or "...and wait for approval first", that is
the signal to move to LangGraph, and this email workflow is exactly that
situation, one real action, sending, that genuinely should not happen
without a human looking at it first.
